In [7]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
conn = sql.connect("database.sqlite")

def sql_query(q):
    return pd.read_sql_query(q, conn)

In [3]:
sql_query("select * from sqlite_master")

,type,name,tbl_name,rootpage,sql
0,table,Salaries,Salaries,2,CREATE TABLE Salaries (\n Id INTEGER PRIMAR...
1,index,salaries_year_idx,Salaries,16566,CREATE INDEX salaries_year_idx ON Salaries (Year)


In [4]:
pd.read_sql_query('select * from sqlite_master',conn)

,type,name,tbl_name,rootpage,sql
0,table,Salaries,Salaries,2,CREATE TABLE Salaries (\n Id INTEGER PRIMAR...
1,index,salaries_year_idx,Salaries,16566,CREATE INDEX salaries_year_idx ON Salaries (Year)


In [5]:
sql_query("select * from sqlite_master where type = 'table'")

,type,name,tbl_name,rootpage,sql
0,table,Salaries,Salaries,2,CREATE TABLE Salaries (\n Id INTEGER PRIMAR...


1. What is the average TotalPayBenefits for each JobTitle? exclude ('Not Provided', 'Not provided')

In [8]:
q = '''SELECT 
    JobTitle, 
    ROUND(AVG(TotalPayBenefits), 2) AS AvgTotalPayBenefits
FROM Salaries
WHERE JobTitle NOT IN ('Not Provided', 'Not provided')
GROUP BY JobTitle
ORDER BY AvgTotalPayBenefits DESC'''
sql_query(q)


,JobTitle,AvgTotalPayBenefits
0,Chief Investment Officer,436224.36
1,Chief of Police,411732.27
2,"Chief, Fire Department",408865.33
3,GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY,399211.28
4,"Gen Mgr, Public Trnsp Dept",380696.44
...,...,...
2153,SPECIAL ASSISTANT XIV,673.80
2154,"BOARD/COMMISSION MEMBER, GROUP III",638.79
2155,"BdComm Mbr, Grp2,M=$25/Mtg",475.05
2156,"BOARD/COMMISSION MEMBER, GROUP II",296.51


2. Which JobTitle has the most employees in each year? exclude ('Not Provided', 'Not provided')

In [9]:
q = '''SELECT Year, JobTitle, EmployeeCount
FROM (
    SELECT 
        Year,
        JobTitle,
        COUNT(*) AS EmployeeCount,
        RANK() OVER (PARTITION BY Year ORDER BY COUNT(*) DESC) AS rnk
    FROM Salaries
    WHERE JobTitle NOT IN ('Not Provided', 'Not provided')
    GROUP BY Year, JobTitle
) AS ranked
WHERE rnk = 1
ORDER BY Year'''

sql_query(q)


,Year,JobTitle,EmployeeCount
0,2011,TRANSIT OPERATOR,2388
1,2012,Transit Operator,2262
2,2013,Transit Operator,2295
3,2014,Transit Operator,2479


3. What is the total number of unique employees? exclude ('Not Provided', 'Not provided')

In [10]:
q = '''
SELECT COUNT(DISTINCT EmployeeName) AS TotalUniqueEmployees
FROM Salaries
WHERE JobTitle NOT IN ('Not Provided', 'Not provided')'''
sql_query(q)


,TotalUniqueEmployees
0,110810


4. Show All Employees Ordered By Their TotalPayBenefits In Descending Order? exclude ('Not Provided', 'Not provided')

In [11]:
q = '''
SELECT *
FROM Salaries
WHERE JobTitle NOT IN ('Not Provided', 'Not provided')
ORDER BY TotalPayBenefits DESC'''
sql_query(q)


,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year,Notes,Agency,Status
0,1,NATHANIEL FORD,GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY,167411.18,0.00,400184.25,,567595.43,567595.43,2011,,San Francisco,
1,2,GARY JIMENEZ,CAPTAIN III (POLICE DEPARTMENT),155966.02,245131.88,137811.38,,538909.28,538909.28,2011,,San Francisco,
2,110532,David Shinn,Deputy Chief 3,129150.01,0.00,342802.63,38780.04,471952.64,510732.68,2014,,San Francisco,PT
3,110533,Amy P Hart,Asst Med Examiner,318835.49,10712.95,60563.54,89540.23,390111.98,479652.21,2014,,San Francisco,FT
4,110534,William J Coaker Jr.,Chief Investment Officer,257340,0.00,82313.70,96570.66,339653.70,436224.36,2014,,San Francisco,PT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
148645,148650,Roy I Tillery,Custodian,0,0.00,0.00,0,0.00,0.00,2014,,San Francisco,PT
148646,110529,Timothy E Gibson,Police Officer 3,,0.00,0.00,-2.73,0.00,-2.73,2013,,San Francisco,
148647,110530,Mark E Laherty,Police Officer 3,,0.00,0.00,-8.2,0.00,-8.20,2013,,San Francisco,
148648,110531,David P Kucia,Police Officer 3,,0.00,0.00,-33.89,0.00,-33.89,2013,,San Francisco,


5. Show All Employees With A TotalPaybenefits Value Between 125,000 and 150,000 And A Job Title 'Firefighter'? exclude ('Not Provided', 'Not provided')

In [12]:
q = '''
SELECT *
FROM Salaries
WHERE JobTitle = 'Firefighter'
  AND TotalPayBenefits BETWEEN 125000 AND 150000
  AND JobTitle NOT IN ('Not Provided', 'Not provided')
  '''
sql_query(q)


,Id,EmployeeName,JobTitle,BasePay,OvertimePay,OtherPay,Benefits,TotalPay,TotalPayBenefits,Year,Notes,Agency,Status
0,44540,Randall Henderson,Firefighter,58564.00,24847.86,43458.06,21042.83,126869.92,147912.75,2012,,San Francisco,
1,44554,Virginia Cheung,Firefighter,101379.60,0.00,12353.42,34129.12,113733.02,147862.14,2012,,San Francisco,
2,44577,Travis Hemenez,Firefighter,83546.57,17897.68,12900.30,33319.57,114344.55,147664.12,2012,,San Francisco,
3,44615,Gail Readdie,Firefighter,83386.40,19715.60,11353.05,32997.12,114455.05,147452.17,2012,,San Francisco,
4,44650,Gregory Ginotti,Firefighter,83546.57,22417.40,8597.70,32654.80,114561.67,147216.47,2012,,San Francisco,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,123134,Stephen J Kloster,Firefighter,80003.76,6747.13,10438.76,29594.48,97189.65,126784.13,2014,,San Francisco,PT
111,123150,Jeffrey D Ryan,Firefighter,72348.75,9340.89,10815.28,34182.21,92504.92,126687.13,2014,,San Francisco,FT
112,123239,Jovan R Blake,Firefighter,72348.75,7747.96,10699.89,35373.02,90796.60,126169.62,2014,,San Francisco,FT
113,123341,Dino M Cafferata,Firefighter,73002.98,5981.80,10944.68,35751.06,89929.46,125680.52,2014,,San Francisco,FT


6. How many employees have no overtime pay or other pay? exclude ('Not Provided', 'Not provided')


In [13]:
q = '''
SELECT COUNT(*) AS EmployeesWithNoExtraPay
FROM Salaries
WHERE (OvertimePay = 0 OR OvertimePay IS NULL)
  AND (OtherPay = 0 OR OtherPay IS NULL)
  AND JobTitle NOT IN ('Not Provided', 'Not provided');
'''
sql_query(q)

,EmployeesWithNoExtraPay
0,34655


7. Show top 10 employees having (overtimepay+OtherPay) in bar plot?  exclude ('Not Provided', 'Not provided')

In [14]:
q = '''
SELECT EmployeeName,
       (OvertimePay + OtherPay) AS ExtraPay
FROM Salaries
WHERE JobTitle NOT IN ('Not Provided', 'Not provided')
ORDER BY ExtraPay DESC
LIMIT 10'''
sql_query(q)


,EmployeeName,ExtraPay
0,NATHANIEL FORD,400184.25
1,GARY JIMENEZ,382943.26
2,David Shinn,342802.63
3,CHRISTOPHER CHONG,254427.61
4,John Goldberg,245999.41
5,Gary Altenberg,234035.79
6,Khoa Trinh,224472.73
7,ALSON LEE,223489.04
8,Brendan A Ward,216637.92
9,Whitney P Yee,213311.84


8. Show The Average of BasePay, OverTime, OtherPay for all employees in pie chart? exclude ('Not Provided', 'Not provided')


In [15]:
q = '''SELECT 
    AVG(BasePay) AS AvgBasePay,
    AVG(OvertimePay) AS AvgOvertimePay,
    AVG(OtherPay) AS AvgOtherPay
FROM Salaries
WHERE JobTitle NOT IN ('Not Provided', 'Not provided')'''

sql_query(q)


,AvgBasePay,AvgOvertimePay,AvgOtherPay
0,66055.506718,5066.059886,3648.767297


9. Get the average TotalPayBenefits for each JobTitle, but only for those with an average pay greater than 75000? exclude ('Not Provided', 'Not provided')


In [16]:
q = '''
SELECT 
    JobTitle, 
    ROUND(AVG(TotalPayBenefits), 2) AS AvgTotalPayBenefits
FROM Salaries
WHERE JobTitle NOT IN ('Not Provided', 'Not provided')
GROUP BY JobTitle
HAVING AVG(TotalPayBenefits) > 75000
ORDER BY AvgTotalPayBenefits DESC'''
sql_query(q)


,JobTitle,AvgTotalPayBenefits
0,Chief Investment Officer,436224.36
1,Chief of Police,411732.27
2,"Chief, Fire Department",408865.33
3,GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY,399211.28
4,"Gen Mgr, Public Trnsp Dept",380696.44
...,...,...
1432,"OPERATING ENGINEER, UNIVERSAL",75273.75
1433,CABLE SPLICER,75255.01
1434,Community Development Asst,75246.66
1435,Senior Legal Process Clerk,75108.72


10. Get the number of employees (EmployeeName) for each JobTitle, where the count is greater than 10?  exclude ('Not Provided', 'Not provided')


In [17]:
q = '''
SELECT 
    JobTitle, 
    COUNT(EmployeeName) AS EmployeeCount
FROM Salaries
WHERE JobTitle NOT IN ('Not Provided', 'Not provided')
GROUP BY JobTitle
HAVING COUNT(EmployeeName) > 10
ORDER BY EmployeeCount DESC'''
sql_query(q)


,JobTitle,EmployeeCount
0,Transit Operator,7036
1,Special Nurse,4389
2,Registered Nurse,3736
3,Public Svc Aide-Public Works,2518
4,Police Officer 3,2421
...,...,...
1151,AUTOMOTIVE MECHANIC ASSISTANT SUPERVISOR,11
1152,ASSOCIATE PERFORMANCE AUDITOR,11
1153,ASSISTANT ELECTRONIC MAINTENANCE TECHNICIAN,11
1154,ARTS PROGRAM ASSISTANT,11
